# Pipeline de Entrenamiento del Autoencoder
Este Jupyter Notebook interactivo te permite extraer datos en tiempo real de **VictoriaMetrics**, realizar el preprocesamiento, entrenar un modelo neuronal de **Autoencoder (Keras)** y establecer los umbrales óptivos para la detección en tiempo real vía **MQTT**.

### 1. Importación de Librerías y Configuración
Cargamos las librerías necesarias y las utilidades del proyecto definitorias de sensores y directorios.

In [1]:
import os
import sys
import shutil
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timezone

# TensorFlow / Keras y sklearn
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.preprocessing import MinMaxScaler

# Utilidades locales
from utils import (
    consultar_prometheus,
    crear_directorios,
    SENSORES,
    COLUMNAS_FEATURES,
    MODELOS_DIR,
    DATOS_DIR,
    estilo_grafica,
)

crear_directorios()
estilo_grafica()
print("[OK] Entorno de Jupyter inicializado correctamente.")


AttributeError: 'OutStream' object has no attribute 'reconfigure'

### 2. Parámetros de Entrenamiento
Configura el rango de fechas para el reentrenamiento diario (por defecto desde el inicio del proyecto `29/05/2026 00:00:00` hasta el momento actual).

In [ ]:
# Edita estos parámetros para entrenar sobre periodos específicos
FECHA_INICIO = "2026-05-29T00:00:00"
FECHA_FIN = None  # None indica el momento actual (fin = datetime.now())
EPOCHS = 100
BATCH_SIZE = 64

print(f"Parámetros listos:")
print(f"  - Fecha Inicio:  {FECHA_INICIO}")
print(f"  - Fecha Fin:     {FECHA_FIN if FECHA_FIN else 'Ahora'}")
print(f"  - Epochs max:    {EPOCHS}")
print(f"  - Batch Size:    {BATCH_SIZE}")


### 3. Extracción de Datos desde VictoriaMetrics
Consultamos a VictoriaMetrics en tiempo real y consolidamos las 6 variables de sensores en un único dataset alineado por timestamp.

In [ ]:
inicio = datetime.fromisoformat(FECHA_INICIO)
fin = datetime.fromisoformat(FECHA_FIN) if FECHA_FIN else datetime.now(timezone.utc)

inicio_naive = inicio.replace(tzinfo=None)
fin_naive = fin.replace(tzinfo=None)

print(f"Extrayendo de VictoriaMetrics:")
print(f"  Intervalo: {inicio_naive} UTC a {fin_naive} UTC\n")

dataframes = {}
for nombre_sensor, config in SENSORES.items():
    print(f"  -> Consultando {nombre_sensor}...")
    df_sensor = consultar_prometheus(
        equipo=config["equipo"],
        metrica=config["metrica"],
        inicio=inicio_naive,
        fin=fin_naive,
        step="15s",
    )
    if not df_sensor.empty:
        dataframes[nombre_sensor] = df_sensor

if len(dataframes) != len(SENSORES):
    faltantes = set(SENSORES.keys()) - set(dataframes.keys())
    raise ValueError(f"No se pudieron extraer todos los sensores. Faltantes: {faltantes}")

# Merge temporal
nombres = list(dataframes.keys())
df_crudo = dataframes[nombres[0]].copy()
for nombre in nombres[1:]:
    df_crudo = pd.merge(df_crudo, dataframes[nombre], on="timestamp", how="outer")

df_crudo = df_crudo.sort_values("timestamp").reset_index(drop=True)
print(f"\n[OK] Extracción exitosa. Filas alineadas: {len(df_crudo):,}")


### 4. Limpieza y Normalización
Imputamos valores nulos utilizando interpolación lineal en sensores analógicos y forward fill para el motor binario. Luego normalizamos a escala `[0, 1]`.

In [ ]:
df = df_crudo.set_index("timestamp").copy()
df = df[COLUMNAS_FEATURES]

col_motor = "MOTOR_01_Running"
cols_continuas = [c for c in df.columns if c != col_motor]

# Imputar NaNs
for col in cols_continuas:
    df[col] = df[col].interpolate(method="linear").ffill().bfill()
if col_motor in df.columns:
    df[col_motor] = df[col_motor].ffill().bfill()

df = df.dropna()
print(f"Muestras limpias totales: {len(df):,}\n")

# Split cronológico 80/20
n_train = int(len(df) * 0.8)
df_train = df.iloc[:n_train].copy()
df_test = df.iloc[n_train:].copy()

print(f"Muestras Entrenamiento: {len(df_train):,}")
print(f"Muestras Validación:    {len(df_test):,}\n")

# Normalizar con MinMaxScaler
scaler = MinMaxScaler(feature_range=(0, 1))
datos_train_norm = scaler.fit_transform(df_train.values)
datos_test_norm = scaler.transform(df_test.values)

df_train_norm = pd.DataFrame(datos_train_norm, columns=df_train.columns, index=df_train.index)
df_test_norm = pd.DataFrame(datos_test_norm, columns=df_test.columns, index=df_test.index)
print("[OK] Datos escalados correctamente.")


### 5. Construcción y Entrenamiento del Autoencoder
Definimos la arquitectura neuronal `6 -> 32 -> 16 -> 8 -> 16 -> 32 -> 6` y la entrenamos utilizando Early Stopping para prevenir sobreajuste.

In [ ]:
n_features = len(COLUMNAS_FEATURES)
inputs = layers.Input(shape=(n_features,), name="input_sensores")

# Encoder
x = layers.Dense(32, activation="relu", name="encoder_1")(inputs)
x = layers.Dense(16, activation="relu", name="encoder_2")(x)
bottleneck = layers.Dense(8, activation="relu", name="bottleneck")(x)

# Decoder
x = layers.Dense(16, activation="relu", name="decoder_1")(bottleneck)
x = layers.Dense(32, activation="relu", name="decoder_2")(x)
outputs = layers.Dense(n_features, activation="sigmoid", name="output_reconstruccion")(x)

autoencoder = keras.Model(inputs=inputs, outputs=outputs, name="autoencoder_anomalias")
autoencoder.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss="mse")

early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True,
    verbose=1,
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=7,
    min_lr=1e-6,
    verbose=1,
)

print("Iniciando entrenamiento...")
history = autoencoder.fit(
    df_train_norm.values.astype(np.float32), df_train_norm.values.astype(np.float32),
    validation_data=(df_test_norm.values.astype(np.float32), df_test_norm.values.astype(np.float32)),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop, reduce_lr],
    shuffle=True,
    verbose=1,
)


### 6. Visualización de Pérdida (Loss)
Graficamos la pérdida del conjunto de entrenamiento y validación para asegurar la convergencia correcta del modelo.

In [ ]:
plt.figure(figsize=(12, 5))
epochs_range = range(1, len(history.history["loss"]) + 1)
plt.plot(epochs_range, history.history["loss"], "b-", label="Loss Train")
plt.plot(epochs_range, history.history["val_loss"], "r-", label="Loss Validación")

best_epoch = np.argmin(history.history["val_loss"]) + 1
best_val = min(history.history["val_loss"])
plt.axvline(x=best_epoch, color="g", linestyle="--", label=f"Mejor Época: {best_epoch}")
plt.scatter([best_epoch], [best_val], color="g", s=100)

plt.title("Curva de Pérdida del Entrenamiento del Autoencoder", fontweight="bold")
plt.xlabel("Época")
plt.ylabel("MSE (Error Medio Cuadrático)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


### 7. Cálculo de Umbrales de Anomalía
Analizamos la distribución de error de reconstrucción sobre el conjunto de entrenamiento. Calculamos y graficamos los percentiles P95 y P99 para delimitar el comportamiento normal.

In [ ]:
X_train = df_train_norm.values.astype(np.float32)
X_reconstructed = autoencoder.predict(X_train, verbose=0)
mse = np.mean(np.square(X_train - X_reconstructed), axis=1)

p95 = float(np.percentile(mse, 95))
p99 = float(np.percentile(mse, 99))
mean = float(np.mean(mse))
std = float(np.std(mse))

print(f"Estadísticas del error de reconstrucción:")
print(f"  - Error Medio:    {mean:.6f}")
print(f"  - Desv. Estándar: {std:.6f}")
print(f"  - Umbral P95:     {p95:.6f} (Falsos positivos teóricos ~5%)")
print(f"  - Umbral P99:     {p99:.6f} (Falsos positivos teóricos ~1%)\n")

# Histograma del error
plt.figure(figsize=(12, 5))
plt.hist(mse, bins=100, color="#2196F3", alpha=0.7, label="Frecuencia de Errores", density=True)
plt.axvline(x=p95, color="orange", linestyle="--", linewidth=2, label=f"P95 ({p95:.6f})")
plt.axvline(x=p99, color="red", linestyle="--", linewidth=2, label=f"P99 ({p99:.6f})")
plt.title("Distribución de Error de Reconstrucción (MSE)", fontweight="bold")
plt.xlabel("MSE Muestra")
plt.ylabel("Densidad")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


### 8. Guardado Seguro de Artefactos de Producción
Si todo es correcto, realizamos un backup del modelo anterior en `modelos/backup/` y guardamos los nuevos artefactos actualizados en la carpeta `modelos/` para su importación inmediata en tiempo real por el script `detectar_mqtt.py`.

In [ ]:
backup_dir = os.path.join(MODELOS_DIR, "backup")
os.makedirs(backup_dir, exist_ok=True)

archivos = {
    "modelo": "autoencoder_anomalias.keras",
    "scaler": "scaler.joblib",
    "umbral": "umbral.joblib",
    "metadata": "metadata_modelo.joblib",
}

# 1. Hacer backup del modelo actual
for k, nombre in archivos.items():
    ruta_orig = os.path.join(MODELOS_DIR, nombre)
    ruta_dest = os.path.join(backup_dir, nombre)
    if os.path.exists(ruta_orig):
        shutil.copy2(ruta_orig, ruta_dest)

print("[OK] Backup resguardado correctamente en modelos/backup/")

# 2. Guardar nuevos archivos en producción
umbral_info = {"p95": p95, "p99": p99, "mean": mean, "std": std}
metadata = {
    "n_features": n_features,
    "columnas": COLUMNAS_FEATURES,
    "arquitectura": "6->32->16->8->16->32->6",
    "loss": "mse",
    "mejor_val_loss": float(best_val),
    "umbral_p95": p95,
    "umbral_p99": p99,
    "train_samples": int(df_train_norm.shape[0]),
    "test_samples": int(df_test_norm.shape[0]),
    "fecha_entrenamiento": datetime.now(timezone.utc).isoformat(),
    "rango_datos": f"{df_crudo['timestamp'].min()} a {df_crudo['timestamp'].max()}",
}

autoencoder.save(os.path.join(MODELOS_DIR, archivos["modelo"]))
joblib.dump(scaler, os.path.join(MODELOS_DIR, archivos["scaler"]))
joblib.dump(umbral_info, os.path.join(MODELOS_DIR, archivos["umbral"]))
joblib.dump(metadata, os.path.join(MODELOS_DIR, archivos["metadata"]))

print("\n=========================================================")
print("  [ÉXITO] ARTEFACTOS ACTUALIZADOS EN PRODUCCIÓN")
print(f"  Rango de datos: {metadata['rango_datos']}")
print(f"  Mejor val_loss: {metadata['mejor_val_loss']:.6f}")
print(f"  Umbral P95:     {metadata['umbral_p95']:.6f}")
print("=========================================================")
